# Demo 00 — Moons, Stars, and Clever Hans

## Did the model learn the shape, or where the shape usually appears?


## Learning goals

Humans see a simple shape-recognition task: classify **moon** versus **star**. The hidden shortcut is absolute position: in the biased training data, moons usually appear near the lower-left corner and stars usually appear near the upper-right.

We will train two models on the same visible task. The MLP looks successful on IID validation but fails when objects move. The CNN is more translation-aware and remains stable. The XAI probes show the difference between learning a shape rule and learning a position shortcut.

The core lesson is deliberately blunt: **the model solved the dataset, not the task**.


## Why this demo matters

Position shortcuts appear in real pipelines through cropping, fixtures, acquisition protocols, conveyor placement, screenshots, camera framing, and labelling workflows. A model can pass ordinary validation if validation repeats the same collection bias.

This generated demo is the no-permission opener for the series. Later notebooks repeat the same audit pattern on industrial images, Waterbirds, and anomaly detection: apparent success first, hidden failure second, explanations third, intervention and re-test last.


### The investigation

1. **The trap:** IID validation says the model works.
2. **The reveal:** moving the same object exposes the shortcut.
3. **The interrogation:** response maps, morphs, paths, and saliency ask what controls the score.
4. **The verdict:** XAI shows the MLP learned where, while the CNN learned a more shape-stable rule.


In [ ]:
# ruff: noqa
from __future__ import annotations

import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display
from PIL import Image, ImageDraw, ImageFilter

SEED = 1701
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

plt.rcParams.update(
    {
        "font.size": 12,
        "axes.titlesize": 15,
        "axes.labelsize": 12,
        "figure.dpi": 140,
        "savefig.dpi": 180,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def find_project_root() -> Path:
    """Return the repository root without assuming the notebook working directory."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the XAI project root.")


def find_notebook_path() -> Path:
    """Locate this notebook from either repo root or notebooks/shortcut_lab."""
    candidates = [
        Path.cwd() / "00_moons_stars_clever_hans.ipynb",
        Path.cwd() / "notebooks/shortcut_lab/00_moons_stars_clever_hans.ipynb",
        find_project_root() / "notebooks/shortcut_lab/00_moons_stars_clever_hans.ipynb",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate 00_moons_stars_clever_hans.ipynb")


PROJECT_ROOT = find_project_root()
NOTEBOOK_PATH = find_notebook_path()
FIGURE_DIR = PROJECT_ROOT / "outputs/notebook_figures/00_moons_stars_clever_hans"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

DEMO = "00 Moons/Stars Clever-Hans position shortcut"
DATA_MODE = "generated_controlled_demo"
EXTERNAL_DATA_REQUIRED = False
MANIFEST_PATH = "not applicable"
MANIFEST_EXISTS = False
DATASET_SOURCE = "Generated inside this notebook"
LICENCE_NOTE = "Repo-authored controlled generated data; no external dataset licence."
MISSING_FILES: list[str] = []
TRUE_TASK = "shape: moon versus star"
SHORTCUT = "absolute object position"
TRAINING_BIAS = "moons lower-left; stars upper-right"
BASELINE_MODEL = "Pixel MLP"
ROBUST_MODEL = "translation-aware CNN"
XAI_PROBES = "movement counterfactuals, position-response maps, shape morphs, decision surfaces, saliency, attribution mass, evidence removal"

status_lines = [
    f"DEMO: {DEMO}",
    f"DATA_MODE: {DATA_MODE}",
    f"EXTERNAL_DATA_REQUIRED: {EXTERNAL_DATA_REQUIRED}",
    f"MANIFEST_PATH: {MANIFEST_PATH}",
    f"MANIFEST_EXISTS: {MANIFEST_EXISTS}",
    f"PROJECT_ROOT: {PROJECT_ROOT}",
    f"DATASET_SOURCE: {DATASET_SOURCE}",
    f"LICENCE_NOTE: {LICENCE_NOTE}",
    f"MISSING_FILES: {MISSING_FILES}",
    f"SEED: {SEED}",
    f"TRUE TASK: {TRUE_TASK}",
    f"SHORTCUT: {SHORTCUT}",
    f"TRAINING BIAS: {TRAINING_BIAS}",
    f"BASELINE MODEL: {BASELINE_MODEL}",
    f"ROBUST MODEL: {ROBUST_MODEL}",
    f"XAI PROBES: {XAI_PROBES}",
]
display(Markdown("```text\n" + "\n".join(status_lines) + "\n```"))


## Dataset and task definition

The true task is shape recognition:

- `moon`
- `star`

The hidden shortcut is absolute position:

- training moons appear in the lower-left quarter, near the corner, with natural variation;
- training stars appear in the upper-right quarter, near the top/right, with natural variation.

The background is neutral and not class-correlated. Shape size, rotation, brightness, stroke, crescent thickness, texture, and within-region position all vary. The paired counterfactual set keeps the same generated object and changes only its position.


In [ ]:
IMAGE_SIZE = 128
MODEL_SIZE = 64
RENDER_SCALE = 3
CLASS_NAMES = {0: "moon", 1: "star"}
CLASS_COLOURS = {0: "#4C78A8", 1: "#F2A541"}
MODEL_COLOURS = {"MLP": "#C44E52", "CNN": "#2E8B57"}
LOWER_LEFT_REGION = (24, 48, 82, 108)  # x_min, x_max, y_min, y_max
UPPER_RIGHT_REGION = (82, 108, 18, 44)
FREE_REGION = (24, 108, 18, 108)
CANONICAL_CENTRES = {"lower_left": (36, 96), "upper_right": (96, 32)}
COMMON_GROUPS = {"moon_lower_left", "star_upper_right"}
CROSSED_GROUPS = {"moon_upper_right", "star_lower_left"}


@dataclass(frozen=True)
class ShapeSpec:
    label: int
    centre: tuple[int, int]
    seed: int
    size: float
    rotation: float
    brightness: int
    stroke: int
    crescent: float


@dataclass(frozen=True)
class PositionSample:
    image: Image.Image
    label: int
    position: str
    centre: tuple[int, int]
    group: str
    split: str
    object_seed: int
    spec: ShapeSpec


def rng_for(seed: int) -> np.random.Generator:
    return np.random.default_rng(seed)


def position_name(centre: tuple[int, int]) -> str:
    x, y = centre
    if x < IMAGE_SIZE // 2 and y >= IMAGE_SIZE // 2:
        return "lower_left"
    if x >= IMAGE_SIZE // 2 and y < IMAGE_SIZE // 2:
        return "upper_right"
    return "uniform"


def group_name(label: int, position: str) -> str:
    return f"{CLASS_NAMES[label]}_{position}"


def choose_centre(region: tuple[int, int, int, int], rng: np.random.Generator) -> tuple[int, int]:
    return (
        int(rng.integers(region[0], region[1] + 1)),
        int(rng.integers(region[2], region[3] + 1)),
    )


def make_shape_spec(
    label: int,
    region: tuple[int, int, int, int],
    seed: int,
    fixed_centre: tuple[int, int] | None = None,
) -> ShapeSpec:
    local_rng = rng_for(seed)
    centre = fixed_centre if fixed_centre is not None else choose_centre(region, local_rng)
    return ShapeSpec(
        label=label,
        centre=centre,
        seed=seed,
        size=float(local_rng.uniform(18.0, 25.0)),
        rotation=float(local_rng.uniform(-20.0, 20.0)),
        brightness=int(local_rng.integers(205, 246)),
        stroke=int(local_rng.integers(1, 3)),
        crescent=float(local_rng.uniform(0.55, 0.74)),
    )


def move_spec(spec: ShapeSpec, centre: tuple[int, int]) -> ShapeSpec:
    return ShapeSpec(
        label=spec.label,
        centre=centre,
        seed=spec.seed,
        size=spec.size,
        rotation=spec.rotation,
        brightness=spec.brightness,
        stroke=spec.stroke,
        crescent=spec.crescent,
    )


def star_points(cx: float, cy: float, outer: float, inner: float, rotation: float) -> list[tuple[float, float]]:
    points = []
    for index in range(10):
        radius = outer if index % 2 == 0 else inner
        angle = math.radians(rotation - 90.0 + index * 36.0)
        points.append((cx + math.cos(angle) * radius, cy + math.sin(angle) * radius))
    return points


def object_mask(spec: ShapeSpec) -> Image.Image:
    canvas_size = IMAGE_SIZE * RENDER_SCALE
    mask = Image.new("L", (canvas_size, canvas_size), 0)
    draw = ImageDraw.Draw(mask)
    cx = spec.centre[0] * RENDER_SCALE
    cy = spec.centre[1] * RENDER_SCALE
    size = spec.size * RENDER_SCALE
    if spec.label == 1:
        draw.polygon(star_points(cx, cy, size, size * 0.45, spec.rotation), fill=255)
    else:
        draw.ellipse((cx - size, cy - size, cx + size, cy + size), fill=255)
        offset = size * spec.crescent
        draw.ellipse((cx - size + offset, cy - size * 1.06, cx + size + offset, cy + size * 1.06), fill=0)
    return mask.filter(ImageFilter.GaussianBlur(0.6 * RENDER_SCALE)).resize(
        (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS
    )


def render_shape(spec: ShapeSpec) -> Image.Image:
    local_rng = rng_for(spec.seed + 99)
    yy, xx = np.mgrid[0:IMAGE_SIZE, 0:IMAGE_SIZE]
    texture = (
        38.0
        + 1.8 * np.sin(xx / 11.0)
        + 1.3 * np.cos(yy / 13.0)
        + local_rng.normal(0.0, 2.2, (IMAGE_SIZE, IMAGE_SIZE))
    )
    background = Image.fromarray(np.clip(texture, 0, 255).astype(np.uint8), "L")
    object_layer = Image.fromarray(np.full((IMAGE_SIZE, IMAGE_SIZE), spec.brightness, dtype=np.uint8), "L")
    return Image.composite(object_layer, background, object_mask(spec))


def make_sample(
    label: int,
    region: tuple[int, int, int, int],
    split: str,
    seed: int,
    fixed_centre: tuple[int, int] | None = None,
) -> PositionSample:
    spec = make_shape_spec(label, region, seed, fixed_centre=fixed_centre)
    position = position_name(spec.centre)
    return PositionSample(
        image=render_shape(spec),
        label=label,
        position=position,
        centre=spec.centre,
        group=group_name(label, position),
        split=split,
        object_seed=seed,
        spec=spec,
    )


def make_dataset(
    split: str,
    common_per_class: int,
    crossed_per_class: int,
    seed_offset: int,
    free_positions: bool = False,
) -> list[PositionSample]:
    samples: list[PositionSample] = []
    counter = seed_offset
    for label in (0, 1):
        common_region = LOWER_LEFT_REGION if label == 0 else UPPER_RIGHT_REGION
        crossed_region = UPPER_RIGHT_REGION if label == 0 else LOWER_LEFT_REGION
        for _ in range(common_per_class):
            region = FREE_REGION if free_positions else common_region
            samples.append(make_sample(label, region, split, SEED + counter))
            counter += 1
        for _ in range(crossed_per_class):
            region = FREE_REGION if free_positions else crossed_region
            samples.append(make_sample(label, region, split, SEED + counter))
            counter += 1
    random.Random(SEED + seed_offset).shuffle(samples)
    return samples


biased_train_samples = make_dataset("biased train", 160, 0, seed_offset=0)
iid_validation_samples = make_dataset("IID validation", 60, 0, seed_offset=10_000)
ood_audit_samples = make_dataset("balanced OOD audit", 60, 60, seed_offset=20_000)
position_augmented_train_samples = make_dataset(
    "position-augmented train", 120, 120, seed_offset=30_000, free_positions=True
)
position_augmented_validation_samples = make_dataset(
    "position-augmented validation", 40, 40, seed_offset=40_000, free_positions=True
)
uniform_audit_samples = make_dataset("uniform-position audit", 80, 0, seed_offset=50_000, free_positions=True)

print(f"Biased training samples: {len(biased_train_samples)}")
print(f"IID validation samples: {len(iid_validation_samples)}")
print(f"Balanced OOD audit samples: {len(ood_audit_samples)}")
print(f"position-augmented train samples: {len(position_augmented_train_samples)}")
print(f"Uniform-position audit samples: {len(uniform_audit_samples)}")


In [ ]:
def save_and_show(fig: plt.Figure, filename: str) -> None:
    fig.savefig(FIGURE_DIR / filename, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)


def image_array(image: Image.Image) -> np.ndarray:
    return np.asarray(image.convert("L"), dtype=np.float32) / 255.0


def representative_sample(label: int, position: str) -> PositionSample:
    centre = CANONICAL_CENTRES[position]
    region = LOWER_LEFT_REGION if position == "lower_left" else UPPER_RIGHT_REGION
    return make_sample(label, region, "representative", SEED + 70_000 + label * 100, fixed_centre=centre)


fig, axes = plt.subplots(2, 2, figsize=(8.4, 7.0), constrained_layout=True)
fig.suptitle("What shortcut exists in the data?", fontsize=18, fontweight="bold")
examples = [
    (0, "lower_left", "common training pattern"),
    (0, "upper_right", "shortcut-breaking"),
    (1, "lower_left", "shortcut-breaking"),
    (1, "upper_right", "common training pattern"),
]
for axis, (label, position, status) in zip(axes.flat, examples, strict=True):
    sample = representative_sample(label, position)
    axis.imshow(sample.image, cmap="gray", vmin=0, vmax=255)
    axis.set_title(f"{CLASS_NAMES[label].title()} at {position.replace('_', '-')}\n{status}")
    axis.set_xticks([])
    axis.set_yticks([])
    colour = "#2E8B57" if status.startswith("common") else "#C44E52"
    for spine in axis.spines.values():
        spine.set_linewidth(3.0)
        spine.set_color(colour)
fig.text(
    0.5,
    -0.02,
    "The true label is the shape. The hidden shortcut is where the shape appears.",
    ha="center",
    fontsize=12,
)
save_and_show(fig, "fig01_shape_position_grid.png")


In [ ]:
def centres_for(samples: list[PositionSample], label: int) -> np.ndarray:
    return np.array([sample.centre for sample in samples if sample.label == label], dtype=float)


fig, axis = plt.subplots(figsize=(7.6, 7.2), constrained_layout=True)
fig.suptitle("Where do moons and stars appear during training?", fontsize=18, fontweight="bold")
for region, label_text, colour in [
    (LOWER_LEFT_REGION, "moon training region", "#4C78A8"),
    (UPPER_RIGHT_REGION, "star training region", "#F2A541"),
]:
    x0, x1, y0, y1 = region
    axis.fill_between([x0, x1], y0, y1, color=colour, alpha=0.12)
    axis.text((x0 + x1) / 2, (y0 + y1) / 2, label_text, ha="center", va="center", color=colour)
for label, marker in [(0, "o"), (1, "*")]:
    pts = centres_for(biased_train_samples, label)
    axis.scatter(
        pts[:, 0],
        pts[:, 1],
        s=42 if label == 0 else 70,
        alpha=0.75,
        marker=marker,
        label=f"training {CLASS_NAMES[label]} centres",
        color=CLASS_COLOURS[label],
        edgecolor="white",
        linewidth=0.5,
    )
for label in (0, 1):
    pts = centres_for(ood_audit_samples, label)
    axis.scatter(pts[:, 0], pts[:, 1], s=12, alpha=0.13, color=CLASS_COLOURS[label])
axis.set_xlim(0, IMAGE_SIZE)
axis.set_ylim(IMAGE_SIZE, 0)
axis.set_xlabel("object centre x")
axis.set_ylabel("object centre y")
axis.set_aspect("equal")
axis.legend(loc="lower right", frameon=False)
axis.set_title("Position predicts the label in the biased split, even though position is not the task.")
save_and_show(fig, "fig02_training_position_distribution.png")


## Model and explanation methods

We compare two learned models.

The baseline is a `PixelMLP`: it flattens the image and learns separate weights for separate pixel locations. It has no architectural reason to treat the same crescent in the lower-left and upper-right as equivalent.

The intervention model is a `GapCNN`: shared convolutional filters scan the image, global average pooling removes the flattened spatial grid before the final linear head, and position augmentation shows the model both shapes across the canvas.

The CNN is not magically translation invariant. It is more translation-aware in this controlled setting because shared filters, global pooling, and position augmentation reduce the reward for absolute location.


In [ ]:
def samples_to_tensors(samples: list[PositionSample]) -> tuple[torch.Tensor, torch.Tensor]:
    arrays = [
        np.asarray(sample.image.resize((MODEL_SIZE, MODEL_SIZE)), dtype=np.float32) / 255.0
        for sample in samples
    ]
    x = torch.tensor(np.stack(arrays)[:, None, :, :], dtype=torch.float32)
    y = torch.tensor([sample.label for sample in samples], dtype=torch.long)
    return x, y


class PixelMLP(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(MODEL_SIZE * MODEL_SIZE, 96),
            torch.nn.ReLU(),
            torch.nn.Linear(96, 48),
            torch.nn.ReLU(),
            torch.nn.Linear(48, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class GapCNN(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(1, 10, kernel_size=5, padding=2),
            torch.nn.ReLU(),
            torch.nn.Conv2d(10, 20, kernel_size=5, padding=2),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(20, 32, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.AdaptiveAvgPool2d(1),
        )
        self.head = torch.nn.Linear(32, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x).flatten(1))


def evaluate_model(model: torch.nn.Module, samples: list[PositionSample]) -> dict[str, Any]:
    x, y = samples_to_tensors(samples)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        loss = torch.nn.functional.cross_entropy(logits, y).item()
        probabilities = torch.softmax(logits, dim=1)
        predictions = logits.argmax(dim=1)
    return {
        "loss": loss,
        "accuracy": float((predictions == y).float().mean().item()),
        "star_score": probabilities[:, 1].cpu().numpy(),
        "predictions": predictions.cpu().numpy(),
        "labels": y.cpu().numpy(),
    }


def train_model(
    model: torch.nn.Module,
    train_samples: list[PositionSample],
    iid_samples: list[PositionSample],
    ood_samples: list[PositionSample],
    epochs: int,
    learning_rate: float,
) -> list[dict[str, float]]:
    x_train, y_train = samples_to_tensors(train_samples)
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    history: list[dict[str, float]] = []
    for epoch in range(1, epochs + 1):
        model.train()
        permutation = torch.randperm(len(train_samples))
        for start in range(0, len(train_samples), 64):
            batch_index = permutation[start : start + 64]
            optimiser.zero_grad()
            loss = torch.nn.functional.cross_entropy(model(x_train[batch_index]), y_train[batch_index])
            loss.backward()
            optimiser.step()
        train_eval = evaluate_model(model, train_samples)
        iid_eval = evaluate_model(model, iid_samples)
        ood_eval = evaluate_model(model, ood_samples)
        history.append(
            {
                "epoch": float(epoch),
                "train_loss": float(train_eval["loss"]),
                "train_accuracy": float(train_eval["accuracy"]),
                "iid_loss": float(iid_eval["loss"]),
                "iid_accuracy": float(iid_eval["accuracy"]),
                "ood_loss": float(ood_eval["loss"]),
                "ood_accuracy": float(ood_eval["accuracy"]),
            }
        )
    return history


baseline_mlp = PixelMLP()
mitigated_cnn = GapCNN()
mlp_history = train_model(
    baseline_mlp,
    biased_train_samples,
    iid_validation_samples,
    ood_audit_samples,
    epochs=22,
    learning_rate=9e-4,
)
cnn_history = train_model(
    mitigated_cnn,
    position_augmented_train_samples,
    position_augmented_validation_samples,
    ood_audit_samples,
    epochs=28,
    learning_rate=1.3e-3,
)
print("Trained MLP on biased positions.")
print("Trained CNN with shared convolution, global average pooling, and position augmentation.")


## Baseline result

Everything initially looks fine. The MLP learns quickly and IID validation is excellent because IID validation repeats the same position bias as training. The audit curve is the warning: when position no longer predicts the label, the MLP does not generalise.


In [ ]:
def plot_history_panel(axis: plt.Axes, history: list[dict[str, float]], model_name: str, metric: str) -> None:
    epochs = [row["epoch"] for row in history]
    suffix = "accuracy" if metric == "accuracy" else "loss"
    axis.plot(epochs, [row[f"train_{suffix}"] for row in history], label="train", linewidth=2.3)
    axis.plot(epochs, [row[f"iid_{suffix}"] for row in history], label="IID validation", linewidth=2.3)
    axis.plot(epochs, [row[f"ood_{suffix}"] for row in history], label="OOD audit", linewidth=2.3)
    axis.set_title(f"{model_name} {metric}")
    axis.set_xlabel("epoch")
    axis.set_ylabel(metric)
    if metric == "accuracy":
        axis.set_ylim(0.0, 1.05)
    axis.grid(alpha=0.18)


fig, axes = plt.subplots(2, 2, figsize=(12.8, 7.6), constrained_layout=True)
fig.suptitle("Everything looks fine — until we ask the right test question", fontsize=18, fontweight="bold")
plot_history_panel(axes[0, 0], mlp_history, "MLP", "accuracy")
plot_history_panel(axes[0, 1], mlp_history, "MLP", "loss")
plot_history_panel(axes[1, 0], cnn_history, "CNN", "accuracy")
plot_history_panel(axes[1, 1], cnn_history, "CNN", "loss")
for axis in axes.flat:
    axis.legend(frameon=False, loc="best")

axes[0, 0].annotate(
    "MLP passes\nIID validation",
    xy=(mlp_history[-1]["epoch"], mlp_history[-1]["iid_accuracy"]),
    xytext=(mlp_history[-1]["epoch"] * 0.45, 0.78),
    arrowprops={"arrowstyle": "->", "color": MODEL_COLOURS["MLP"], "lw": 1.6},
    color=MODEL_COLOURS["MLP"],
    fontweight="bold",
)
axes[0, 1].annotate(
    "confidently wrong\non OOD audit",
    xy=(mlp_history[-1]["epoch"], mlp_history[-1]["ood_loss"]),
    xytext=(mlp_history[-1]["epoch"] * 0.28, max(row["ood_loss"] for row in mlp_history) * 0.72),
    arrowprops={"arrowstyle": "->", "color": MODEL_COLOURS["MLP"], "lw": 1.6},
    color=MODEL_COLOURS["MLP"],
    fontweight="bold",
)
axes[1, 0].annotate(
    "CNN improves\nOOD audit",
    xy=(cnn_history[-1]["epoch"], cnn_history[-1]["ood_accuracy"]),
    xytext=(cnn_history[-1]["epoch"] * 0.42, 0.62),
    arrowprops={"arrowstyle": "->", "color": MODEL_COLOURS["CNN"], "lw": 1.6},
    color=MODEL_COLOURS["CNN"],
    fontweight="bold",
)
fig.text(
    0.5,
    -0.02,
    "Ordinary validation repeats the position bias. The audit breaks it.",
    ha="center",
    fontsize=12,
    fontweight="bold",
)
save_and_show(fig, "fig03_training_curves.png")


## Failure or pitfall

The audit asks a different question from ordinary validation: what happens when the object appears somewhere the shortcut does not expect? This is where the hidden failure becomes visible.


In [ ]:
def group_accuracies(model: torch.nn.Module, samples: list[PositionSample]) -> dict[str, float]:
    evaluation = evaluate_model(model, samples)
    predictions = evaluation["predictions"]
    labels = evaluation["labels"]
    groups = np.array([sample.group for sample in samples])
    return {
        group: float((predictions[groups == group] == labels[groups == group]).mean())
        for group in sorted(set(groups))
    }


def subset_accuracy(model: torch.nn.Module, samples: list[PositionSample], groups: set[str]) -> float:
    selected = [sample for sample in samples if sample.group in groups]
    return float(evaluate_model(model, selected)["accuracy"])


mlp_train_accuracy = float(evaluate_model(baseline_mlp, biased_train_samples)["accuracy"])
mlp_iid_validation_accuracy = float(evaluate_model(baseline_mlp, iid_validation_samples)["accuracy"])
mlp_ood_accuracy = float(evaluate_model(baseline_mlp, ood_audit_samples)["accuracy"])
mlp_common_accuracy = subset_accuracy(baseline_mlp, ood_audit_samples, COMMON_GROUPS)
mlp_crossed_accuracy = subset_accuracy(baseline_mlp, ood_audit_samples, CROSSED_GROUPS)
mlp_uniform_accuracy = float(evaluate_model(baseline_mlp, uniform_audit_samples)["accuracy"])
mlp_group_accuracies = group_accuracies(baseline_mlp, ood_audit_samples)
mlp_worst_group_accuracy = min(mlp_group_accuracies.values())

cnn_train_accuracy = float(evaluate_model(mitigated_cnn, position_augmented_train_samples)["accuracy"])
cnn_iid_validation_accuracy = float(evaluate_model(mitigated_cnn, position_augmented_validation_samples)["accuracy"])
cnn_ood_accuracy = float(evaluate_model(mitigated_cnn, ood_audit_samples)["accuracy"])
cnn_common_accuracy = subset_accuracy(mitigated_cnn, ood_audit_samples, COMMON_GROUPS)
cnn_crossed_accuracy = subset_accuracy(mitigated_cnn, ood_audit_samples, CROSSED_GROUPS)
cnn_uniform_accuracy = float(evaluate_model(mitigated_cnn, uniform_audit_samples)["accuracy"])
cnn_group_accuracies = group_accuracies(mitigated_cnn, ood_audit_samples)
cnn_worst_group_accuracy = min(cnn_group_accuracies.values())


def fixed_pair(label: int, usual_position: str, moved_position: str, seed: int) -> tuple[PositionSample, PositionSample]:
    usual_region = LOWER_LEFT_REGION if usual_position == "lower_left" else UPPER_RIGHT_REGION
    usual_spec = make_shape_spec(label, usual_region, seed, fixed_centre=CANONICAL_CENTRES[usual_position])
    moved_spec = move_spec(usual_spec, CANONICAL_CENTRES[moved_position])
    usual = PositionSample(
        render_shape(usual_spec),
        label,
        usual_position,
        usual_spec.centre,
        group_name(label, usual_position),
        "paired counterfactual",
        seed,
        usual_spec,
    )
    moved = PositionSample(
        render_shape(moved_spec),
        label,
        moved_position,
        moved_spec.centre,
        group_name(label, moved_position),
        "paired counterfactual",
        seed,
        moved_spec,
    )
    return usual, moved


def make_counterfactual_pairs(count: int) -> list[tuple[PositionSample, PositionSample]]:
    pairs: list[tuple[PositionSample, PositionSample]] = []
    for index in range(count):
        pairs.append(fixed_pair(0, "lower_left", "upper_right", SEED + 80_000 + index))
        pairs.append(fixed_pair(1, "upper_right", "lower_left", SEED + 81_000 + index))
    return pairs


def counterfactual_flip_rate(model: torch.nn.Module, pairs: list[tuple[PositionSample, PositionSample]]) -> float:
    flips = 0
    for usual, moved in pairs:
        predictions = evaluate_model(model, [usual, moved])["predictions"]
        flips += int(predictions[0] != predictions[1])
    return flips / len(pairs)


counterfactual_pairs = make_counterfactual_pairs(30)
mlp_counterfactual_flip_rate = counterfactual_flip_rate(baseline_mlp, counterfactual_pairs)
cnn_counterfactual_flip_rate = counterfactual_flip_rate(mitigated_cnn, counterfactual_pairs)
mlp_shortcut_gap = mlp_common_accuracy - mlp_crossed_accuracy
cnn_shortcut_gap = cnn_common_accuracy - cnn_crossed_accuracy

performance_metrics = [
    ("Train", mlp_train_accuracy, cnn_train_accuracy),
    ("IID\nvalidation", mlp_iid_validation_accuracy, cnn_iid_validation_accuracy),
    ("Balanced\nOOD", mlp_ood_accuracy, cnn_ood_accuracy),
    ("Worst\nOOD group", mlp_worst_group_accuracy, cnn_worst_group_accuracy),
    ("Shortcut-\nbreaking", mlp_crossed_accuracy, cnn_crossed_accuracy),
    ("Uniform\nposition", mlp_uniform_accuracy, cnn_uniform_accuracy),
]
risk_metrics = [
    ("Movement\nflip rate", mlp_counterfactual_flip_rate, cnn_counterfactual_flip_rate),
    ("Common-crossed\ngap", mlp_shortcut_gap, cnn_shortcut_gap),
]

fig = plt.figure(figsize=(13.4, 6.7), constrained_layout=True)
grid = fig.add_gridspec(2, 3, width_ratios=[1.25, 2.2, 1.25], height_ratios=[1.0, 1.0])
fig.suptitle("Which model learned shape, and which learned position?", fontsize=18, fontweight="bold")
callout_axis = fig.add_subplot(grid[:, 0])
callout_axis.axis("off")
callout_axis.add_patch(plt.Rectangle((0.02, 0.06), 0.94, 0.88, facecolor="#F7F7F7", edgecolor="#D0D0D0", linewidth=1.2))
callout_axis.text(0.08, 0.82, "Metric trap", fontsize=15, fontweight="bold", transform=callout_axis.transAxes)
callout_axis.text(
    0.08,
    0.62,
    "The MLP passes\nordinary validation\nbut fails the\nshortcut-breaking audit.",
    fontsize=12,
    color=MODEL_COLOURS["MLP"],
    fontweight="bold",
    transform=callout_axis.transAxes,
)
callout_axis.text(
    0.08,
    0.30,
    "The CNN keeps\nits answer more\nstable when\nposition changes.",
    fontsize=12,
    color=MODEL_COLOURS["CNN"],
    fontweight="bold",
    transform=callout_axis.transAxes,
)

perf_axis = fig.add_subplot(grid[:, 1])
x = np.arange(len(performance_metrics))
width = 0.36
perf_axis.bar(x - width / 2, [row[1] for row in performance_metrics], width, color=MODEL_COLOURS["MLP"], label="MLP")
perf_axis.bar(x + width / 2, [row[2] for row in performance_metrics], width, color=MODEL_COLOURS["CNN"], label="CNN")
perf_axis.set_title("Performance: higher is better")
perf_axis.set_ylabel("accuracy")
perf_axis.set_ylim(0, 1.08)
perf_axis.set_xticks(x)
perf_axis.set_xticklabels([row[0] for row in performance_metrics])
perf_axis.grid(axis="y", alpha=0.18)
perf_axis.legend(frameon=False, loc="lower left")
for offset, values in [(-width / 2, [row[1] for row in performance_metrics]), (width / 2, [row[2] for row in performance_metrics])]:
    for xpos, value in zip(x + offset, values, strict=True):
        perf_axis.text(xpos, value + 0.025, f"{value * 100:.0f}%", ha="center", fontsize=8.5)

risk_axis = fig.add_subplot(grid[:, 2])
risk_y = np.arange(len(risk_metrics))
risk_axis.barh(risk_y - 0.18, [row[1] for row in risk_metrics], 0.32, color=MODEL_COLOURS["MLP"], label="MLP")
risk_axis.barh(risk_y + 0.18, [row[2] for row in risk_metrics], 0.32, color=MODEL_COLOURS["CNN"], label="CNN")
risk_axis.set_title("Shortcut risk: lower is better")
risk_axis.set_xlim(0, 1.05)
risk_axis.set_yticks(risk_y)
risk_axis.set_yticklabels([row[0] for row in risk_metrics])
risk_axis.grid(axis="x", alpha=0.18)
for offset, values in [(-0.18, [row[1] for row in risk_metrics]), (0.18, [row[2] for row in risk_metrics])]:
    for ypos, value in zip(risk_y + offset, values, strict=True):
        risk_axis.text(value + 0.03, ypos, f"{value * 100:.0f}%", va="center", fontsize=9)
fig.text(
    0.5,
    -0.02,
    "Accuracy and risk are separated because high accuracy is good, while high movement sensitivity is bad.",
    ha="center",
    fontsize=12,
)
save_and_show(fig, "fig04_metric_dashboard.png")


### Same-shape counterfactual movement

This is the central Clever-Hans test. We keep the object seed fixed, keep the same shape, size, rotation, brightness, and crescent or star geometry, and change only the position.

A shape-based model should keep the label when the object moves. A position-shortcut model will change its decision with the region.


In [ ]:
moon_lower_left, moon_upper_right = fixed_pair(0, "lower_left", "upper_right", SEED + 92_001)
star_upper_right, star_lower_left = fixed_pair(1, "upper_right", "lower_left", SEED + 92_011)
selected_counterfactual_samples = [moon_lower_left, moon_upper_right, star_upper_right, star_lower_left]
selected_mlp_eval = evaluate_model(baseline_mlp, selected_counterfactual_samples)
selected_cnn_eval = evaluate_model(mitigated_cnn, selected_counterfactual_samples)
mlp_cf_predictions = [int(value) for value in selected_mlp_eval["predictions"]]
cnn_cf_predictions = [int(value) for value in selected_cnn_eval["predictions"]]
cf_scores = {"MLP": selected_mlp_eval["star_score"], "CNN": selected_cnn_eval["star_score"]}

fig, axes = plt.subplots(2, 2, figsize=(11.4, 7.4), constrained_layout=True)
fig.suptitle("Same shape, different position, different prediction", fontsize=19, fontweight="bold")
hero_panels = [
    (axes[0, 0], "Moon lower-left", moon_lower_left, 0),
    (axes[0, 1], "Same moon upper-right", moon_upper_right, 1),
    (axes[1, 0], "Star upper-right", star_upper_right, 2),
    (axes[1, 1], "Same star lower-left", star_lower_left, 3),
]
for axis, title, sample, idx in hero_panels:
    axis.imshow(sample.image, cmap="gray", vmin=0, vmax=255)
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_title(f"{title}\nTRUE: {CLASS_NAMES[sample.label]} | POSITION: {sample.position.replace('_', '-')}", fontsize=11.5)
    mlp_pred = mlp_cf_predictions[idx]
    cnn_pred = cnn_cf_predictions[idx]
    mlp_ok = mlp_pred == sample.label
    cnn_ok = cnn_pred == sample.label
    mlp_score = cf_scores["MLP"][idx] if mlp_pred == 1 else 1.0 - cf_scores["MLP"][idx]
    cnn_score = cf_scores["CNN"][idx] if cnn_pred == 1 else 1.0 - cf_scores["CNN"][idx]
    badge_specs = [
        (0.04, f"MLP: {CLASS_NAMES[mlp_pred]}, score {mlp_score:.2f} {'OK' if mlp_ok else 'FAIL'}", MODEL_COLOURS["MLP"], mlp_ok),
        (0.55, f"CNN: {CLASS_NAMES[cnn_pred]}, score {cnn_score:.2f} {'OK' if cnn_ok else 'FAIL'}", MODEL_COLOURS["CNN"], cnn_ok),
    ]
    for x0, text, colour, ok in badge_specs:
        axis.text(
            x0,
            -0.13,
            text,
            transform=axis.transAxes,
            fontsize=10.5,
            color="white",
            fontweight="bold",
            bbox={"boxstyle": "round,pad=0.35", "facecolor": colour if ok else "#B23A48", "edgecolor": "none", "alpha": 0.95},
        )
fig.text(0.5, -0.02, "The MLP learned where, not what.", ha="center", fontsize=14, fontweight="bold", color=MODEL_COLOURS["MLP"])
save_and_show(fig, "fig05_same_shape_moved.png")

selected_mlp_moon_counterfactual_flipped = mlp_cf_predictions[0] != mlp_cf_predictions[1]
selected_cnn_moon_counterfactual_stable = cnn_cf_predictions[0] == cnn_cf_predictions[1] == 0
selected_mlp_star_counterfactual_flipped = mlp_cf_predictions[2] != mlp_cf_predictions[3]
selected_cnn_star_counterfactual_stable = cnn_cf_predictions[2] == cnn_cf_predictions[3] == 1


## Position-response maps

For a fixed moon and a fixed star, we move the same object across the canvas and record the model's star score. The object shape is fixed. Only location changes.

This is the main XAI probe for this demo: a position-shortcut model changes its prediction across the canvas even when the object is unchanged.


In [ ]:
SCAN_VALUES = np.linspace(22, 106, 25).astype(int)


def sample_from_spec(spec: ShapeSpec, centre: tuple[int, int], split: str = "scan") -> PositionSample:
    moved = move_spec(spec, centre)
    position = position_name(centre)
    return PositionSample(render_shape(moved), moved.label, position, centre, group_name(moved.label, position), split, moved.seed, moved)


fixed_moon_spec = moon_lower_left.spec
fixed_star_spec = star_upper_right.spec


def position_response(model: torch.nn.Module, spec: ShapeSpec) -> np.ndarray:
    samples = [sample_from_spec(spec, (int(x), int(y))) for y in SCAN_VALUES for x in SCAN_VALUES]
    scores = evaluate_model(model, samples)["star_score"]
    return scores.reshape(len(SCAN_VALUES), len(SCAN_VALUES))


moon_position_response_mlp = position_response(baseline_mlp, fixed_moon_spec)
star_position_response_mlp = position_response(baseline_mlp, fixed_star_spec)
moon_position_response_cnn = position_response(mitigated_cnn, fixed_moon_spec)
star_position_response_cnn = position_response(mitigated_cnn, fixed_star_spec)


def sensitivity_index(matrix: np.ndarray) -> float:
    return float(np.max(matrix) - np.min(matrix))


def shape_consistency(matrix: np.ndarray, true_label: int) -> float:
    predictions = (matrix >= 0.5).astype(int)
    return float(np.mean(predictions == true_label))


position_response_metrics = {
    "mlp_moon_psi": sensitivity_index(moon_position_response_mlp),
    "mlp_star_psi": sensitivity_index(star_position_response_mlp),
    "cnn_moon_psi": sensitivity_index(moon_position_response_cnn),
    "cnn_star_psi": sensitivity_index(star_position_response_cnn),
    "mlp_moon_shape_consistency": shape_consistency(moon_position_response_mlp, 0),
    "mlp_star_shape_consistency": shape_consistency(star_position_response_mlp, 1),
    "cnn_moon_shape_consistency": shape_consistency(moon_position_response_cnn, 0),
    "cnn_star_shape_consistency": shape_consistency(star_position_response_cnn, 1),
}
mlp_position_sensitivity = float(np.mean([position_response_metrics["mlp_moon_psi"], position_response_metrics["mlp_star_psi"]]))
cnn_position_sensitivity = float(np.mean([position_response_metrics["cnn_moon_psi"], position_response_metrics["cnn_star_psi"]]))
mlp_shape_consistency = float(np.mean([position_response_metrics["mlp_moon_shape_consistency"], position_response_metrics["mlp_star_shape_consistency"]]))
cnn_shape_consistency = float(np.mean([position_response_metrics["cnn_moon_shape_consistency"], position_response_metrics["cnn_star_shape_consistency"]]))

response_panels = [
    ("MLP fixed moon", moon_position_response_mlp, 0, "star score rises\nin upper-right"),
    ("MLP fixed star", star_position_response_mlp, 1, "star score falls\nin lower-left"),
    ("CNN fixed moon", moon_position_response_cnn, 0, "stable moon\nprediction"),
    ("CNN fixed star", star_position_response_cnn, 1, "stable star\nprediction"),
]
fig, axes = plt.subplots(2, 2, figsize=(11.4, 9.2), constrained_layout=True)
fig.suptitle("What happens when we move the same object around the image?", fontsize=18, fontweight="bold")
for axis, (title, matrix, true_label, annotation) in zip(axes.flat, response_panels, strict=True):
    im = axis.imshow(
        matrix,
        cmap="RdYlBu_r",
        vmin=0,
        vmax=1,
        extent=[SCAN_VALUES[0], SCAN_VALUES[-1], SCAN_VALUES[-1], SCAN_VALUES[0]],
        interpolation="bicubic",
    )
    axis.contour(SCAN_VALUES, SCAN_VALUES, matrix, levels=[0.5], colors="black", linewidths=1.4)
    axis.add_patch(plt.Rectangle((LOWER_LEFT_REGION[0], LOWER_LEFT_REGION[2]), LOWER_LEFT_REGION[1]-LOWER_LEFT_REGION[0], LOWER_LEFT_REGION[3]-LOWER_LEFT_REGION[2], fill=False, edgecolor=CLASS_COLOURS[0], linewidth=2))
    axis.add_patch(plt.Rectangle((UPPER_RIGHT_REGION[0], UPPER_RIGHT_REGION[2]), UPPER_RIGHT_REGION[1]-UPPER_RIGHT_REGION[0], UPPER_RIGHT_REGION[3]-UPPER_RIGHT_REGION[2], fill=False, edgecolor=CLASS_COLOURS[1], linewidth=2))
    psi = sensitivity_index(matrix)
    consistency = shape_consistency(matrix, true_label)
    axis.set_title(f"{title}\nPSI {psi:.2f}; shape consistency {consistency * 100:.0f}%")
    axis.text(0.04, 0.08, annotation, transform=axis.transAxes, fontsize=10.5, fontweight="bold", color="white", bbox={"boxstyle":"round,pad=0.35", "facecolor":"black", "alpha":0.70, "edgecolor":"none"})
    axis.set_xlabel("object centre x")
    axis.set_ylabel("object centre y")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.82, label="predicted probability of star")
fig.text(0.18, 0.015, "blue = moon-like score", color="#2D6F9F", fontsize=11, fontweight="bold")
fig.text(0.38, 0.015, "red = star-like score", color="#B23A48", fontsize=11, fontweight="bold")
fig.text(0.66, 0.015, "PSI is score range; shape consistency is the share of positions classified as the true shape.", fontsize=10.5)
save_and_show(fig, "fig06_position_response_maps.png")


## Decision boundary in position space

The response map can also be read as a decision boundary. A boundary that slices the canvas by location is evidence of a spatial template, not a shape rule.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.4, 9.0), constrained_layout=True)
fig.suptitle("Is the decision boundary about shape or position?", fontsize=18, fontweight="bold")
boundary_panels = [
    ("MLP: same moon", moon_position_response_mlp, moon_lower_left.centre, moon_upper_right.centre),
    ("CNN: same moon", moon_position_response_cnn, moon_lower_left.centre, moon_upper_right.centre),
    ("MLP: same star", star_position_response_mlp, star_upper_right.centre, star_lower_left.centre),
    ("CNN: same star", star_position_response_cnn, star_upper_right.centre, star_lower_left.centre),
]
decision_boundary_results = {
    "mlp_moon_crosses": bool(np.any(moon_position_response_mlp >= 0.5) and np.any(moon_position_response_mlp < 0.5)),
    "mlp_star_crosses": bool(np.any(star_position_response_mlp >= 0.5) and np.any(star_position_response_mlp < 0.5)),
    "cnn_moon_consistency": shape_consistency(moon_position_response_cnn, 0),
    "cnn_star_consistency": shape_consistency(star_position_response_cnn, 1),
}
for axis, (title, matrix, start_centre, end_centre) in zip(axes.flat, boundary_panels, strict=True):
    axis.contourf(SCAN_VALUES, SCAN_VALUES, matrix, levels=np.linspace(0, 1, 15), cmap="RdYlBu_r", alpha=0.92)
    axis.contour(SCAN_VALUES, SCAN_VALUES, matrix, levels=[0.5], colors="black", linewidths=1.8)
    axis.annotate("", xy=end_centre, xytext=start_centre, arrowprops={"arrowstyle":"->", "color":"black", "lw":2.2, "linestyle":"--"})
    axis.scatter([start_centre[0]], [start_centre[1]], s=75, color="#2E8B57", zorder=5)
    axis.scatter([end_centre[0]], [end_centre[1]], s=75, color="#C44E52", zorder=5)
    axis.text(start_centre[0] + 2, start_centre[1] + 4, "usual", color="#2E8B57", fontweight="bold")
    axis.text(end_centre[0] + 2, end_centre[1] - 4, "moved", color="#C44E52", fontweight="bold")
    axis.add_patch(plt.Rectangle((LOWER_LEFT_REGION[0], LOWER_LEFT_REGION[2]), LOWER_LEFT_REGION[1]-LOWER_LEFT_REGION[0], LOWER_LEFT_REGION[3]-LOWER_LEFT_REGION[2], fill=False, edgecolor=CLASS_COLOURS[0], linewidth=2))
    axis.add_patch(plt.Rectangle((UPPER_RIGHT_REGION[0], UPPER_RIGHT_REGION[2]), UPPER_RIGHT_REGION[1]-UPPER_RIGHT_REGION[0], UPPER_RIGHT_REGION[3]-UPPER_RIGHT_REGION[2], fill=False, edgecolor=CLASS_COLOURS[1], linewidth=2))
    axis.set_title(title)
    axis.set_xlim(SCAN_VALUES[0], SCAN_VALUES[-1])
    axis.set_ylim(SCAN_VALUES[-1], SCAN_VALUES[0])
    axis.set_xlabel("object centre x")
    axis.set_ylabel("object centre y")
fig.text(0.5, -0.02, "For the MLP, the boundary cuts the canvas by location. For the CNN, the decision is much more stable for the same shape across positions.", ha="center", fontsize=12)
save_and_show(fig, "fig07_position_decision_boundary.png")


## Shape-morph counterfactuals

Movement probes test whether position changes the answer. Shape morphs ask the complementary question: when position is fixed, does the score respond to shape?

We blend a generated moon image into a generated star image at the same position. This is a diagnostic interpolation, not a claim about natural image morphing.


In [ ]:
def render_morph_image(alpha: float, centre: tuple[int, int], seed: int) -> Image.Image:
    moon_spec = make_shape_spec(0, FREE_REGION, seed, fixed_centre=centre)
    star_spec = ShapeSpec(
        label=1,
        centre=centre,
        seed=seed,
        size=moon_spec.size,
        rotation=moon_spec.rotation + 8.0,
        brightness=moon_spec.brightness,
        stroke=moon_spec.stroke,
        crescent=moon_spec.crescent,
    )
    moon = np.asarray(render_shape(moon_spec), dtype=np.float32)
    star = np.asarray(render_shape(star_spec), dtype=np.float32)
    blended = np.clip((1.0 - alpha) * moon + alpha * star, 0, 255).astype(np.uint8)
    return Image.fromarray(blended, "L")


def score_image(model: torch.nn.Module, image: Image.Image) -> float:
    sample = PositionSample(image, 0, "diagnostic", (0, 0), "diagnostic", "diagnostic", SEED, fixed_moon_spec)
    return float(evaluate_model(model, [sample])["star_score"][0])


morph_alphas = np.linspace(0, 1, 31)
strip_alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
position_alphas = np.linspace(0, 1, 25)

shape_morph_results: dict[str, dict[str, np.ndarray]] = {}
for position_name_key, centre in CANONICAL_CENTRES.items():
    morph_images = [render_morph_image(float(alpha), centre, SEED + 93_000) for alpha in morph_alphas]
    shape_morph_results[position_name_key] = {
        "mlp": np.array([score_image(baseline_mlp, image) for image in morph_images]),
        "cnn": np.array([score_image(mitigated_cnn, image) for image in morph_images]),
    }

fig, axes = plt.subplots(2, 5, figsize=(13.2, 6.2), constrained_layout=True)
fig.suptitle("Does the score follow shape or position?", fontsize=18, fontweight="bold")
for row, (position_name_key, row_title) in enumerate([("lower_left", "lower-left"), ("upper_right", "upper-right")]):
    centre = CANONICAL_CENTRES[position_name_key]
    for col, alpha in enumerate(strip_alphas):
        axis = axes[row, col]
        image = render_morph_image(alpha, centre, SEED + 93_000)
        nearest_index = int(round(alpha * (len(morph_alphas) - 1)))
        mlp_score = shape_morph_results[position_name_key]["mlp"][nearest_index]
        cnn_score = shape_morph_results[position_name_key]["cnn"][nearest_index]
        axis.imshow(image, cmap="gray", vmin=0, vmax=255)
        axis.set_xticks([])
        axis.set_yticks([])
        if row == 0:
            axis.set_title(["moon", "moon-ish", "ambiguous", "star-ish", "star"][col], fontsize=11)
        axis.text(
            0.5,
            -0.11,
            f"{row_title}\nMLP star {mlp_score:.2f}\nCNN star {cnn_score:.2f}",
            transform=axis.transAxes,
            ha="center",
            va="top",
            fontsize=9.5,
            bbox={"boxstyle":"round,pad=0.25", "facecolor":"white", "edgecolor":"#DDDDDD", "alpha":0.95},
        )
fig.text(0.5, -0.03, "At upper-right, the MLP can score an object as star-like before it looks star-like. The CNN tracks the morph more consistently.", ha="center", fontsize=12, fontweight="bold")
save_and_show(fig, "fig08_shape_morph_strip.png")

shape_position_surface_mlp = np.zeros((len(morph_alphas), len(position_alphas)))
shape_position_surface_cnn = np.zeros_like(shape_position_surface_mlp)
for row, shape_alpha in enumerate(morph_alphas):
    for col, position_alpha in enumerate(position_alphas):
        x = int((1.0 - position_alpha) * CANONICAL_CENTRES["lower_left"][0] + position_alpha * CANONICAL_CENTRES["upper_right"][0])
        y = int((1.0 - position_alpha) * CANONICAL_CENTRES["lower_left"][1] + position_alpha * CANONICAL_CENTRES["upper_right"][1])
        morph_image = render_morph_image(float(shape_alpha), (x, y), SEED + 93_000)
        shape_position_surface_mlp[row, col] = score_image(baseline_mlp, morph_image)
        shape_position_surface_cnn[row, col] = score_image(mitigated_cnn, morph_image)

fig, axes = plt.subplots(1, 2, figsize=(12.2, 5.4), constrained_layout=True)
fig.suptitle("Shape versus position: what controls the score?", fontsize=18, fontweight="bold")
for axis, title, surface, note in [
    (axes[0], "MLP", shape_position_surface_mlp, "position dominates"),
    (axes[1], "CNN", shape_position_surface_cnn, "shape dominates"),
]:
    im = axis.imshow(surface, cmap="RdYlBu_r", vmin=0, vmax=1, origin="lower", aspect="auto", extent=[0, 1, 0, 1], interpolation="bicubic")
    axis.contour(position_alphas, morph_alphas, surface, levels=[0.5], colors="black", linewidths=1.5)
    axis.text(0.04, 0.88, note, transform=axis.transAxes, fontsize=12, fontweight="bold", color="white", bbox={"boxstyle":"round,pad=0.35", "facecolor":"black", "alpha":0.72, "edgecolor":"none"})
    axis.set_title(title)
    axis.set_xlabel("position: lower-left → upper-right")
    axis.set_ylabel("shape: moon → star")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.88, label="predicted probability of star")
fig.text(0.5, -0.03, "A vertical boundary means position controls the score. A horizontal boundary means shape controls the score.", ha="center", fontsize=12)
save_and_show(fig, "fig09_shape_position_surface.png")


## Counterfactual change: movement path

Now we ask a direct operational question: how far must the object move before the model changes its mind?


In [ ]:
def interpolate_centres(start: tuple[int, int], end: tuple[int, int], steps: int = 31) -> list[tuple[int, int]]:
    centres = []
    for alpha in np.linspace(0, 1, steps):
        x = int(round((1.0 - alpha) * start[0] + alpha * end[0]))
        y = int(round((1.0 - alpha) * start[1] + alpha * end[1]))
        centres.append((x, y))
    return centres


def movement_samples(spec: ShapeSpec, start: tuple[int, int], end: tuple[int, int]) -> list[PositionSample]:
    return [sample_from_spec(spec, centre, split="movement path") for centre in interpolate_centres(start, end)]


def movement_scores(model: torch.nn.Module, spec: ShapeSpec, start: tuple[int, int], end: tuple[int, int]) -> np.ndarray:
    return evaluate_model(model, movement_samples(spec, start, end))["star_score"]


movement_progress = np.linspace(0, 1, 31)
moon_movement_samples = movement_samples(fixed_moon_spec, CANONICAL_CENTRES["lower_left"], CANONICAL_CENTRES["upper_right"])
star_movement_samples = movement_samples(fixed_star_spec, CANONICAL_CENTRES["upper_right"], CANONICAL_CENTRES["lower_left"])
movement_path_results = {
    "moon_mlp": movement_scores(baseline_mlp, fixed_moon_spec, CANONICAL_CENTRES["lower_left"], CANONICAL_CENTRES["upper_right"]),
    "moon_cnn": movement_scores(mitigated_cnn, fixed_moon_spec, CANONICAL_CENTRES["lower_left"], CANONICAL_CENTRES["upper_right"]),
    "star_mlp": movement_scores(baseline_mlp, fixed_star_spec, CANONICAL_CENTRES["upper_right"], CANONICAL_CENTRES["lower_left"]),
    "star_cnn": movement_scores(mitigated_cnn, fixed_star_spec, CANONICAL_CENTRES["upper_right"], CANONICAL_CENTRES["lower_left"]),
}


def first_crossing(progress: np.ndarray, scores: np.ndarray, starts_above: bool) -> float | None:
    if starts_above:
        crossed = np.where(scores < 0.5)[0]
    else:
        crossed = np.where(scores >= 0.5)[0]
    return None if len(crossed) == 0 else float(progress[int(crossed[0])])


fig = plt.figure(figsize=(13.4, 7.6), constrained_layout=True)
grid = fig.add_gridspec(3, 10, height_ratios=[2.3, 0.95, 0.95])
fig.suptitle("How far must the object move before the model changes its mind?", fontsize=18, fontweight="bold")
for axis, title, mlp_key, cnn_key, starts_above in [
    (fig.add_subplot(grid[0, :5]), "Same moon: lower-left → upper-right", "moon_mlp", "moon_cnn", False),
    (fig.add_subplot(grid[0, 5:]), "Same star: upper-right → lower-left", "star_mlp", "star_cnn", True),
]:
    axis.plot(movement_progress, movement_path_results[mlp_key], label="MLP", color=MODEL_COLOURS["MLP"], linewidth=2.6)
    axis.plot(movement_progress, movement_path_results[cnn_key], label="CNN", color=MODEL_COLOURS["CNN"], linewidth=2.6)
    axis.axhline(0.5, color="black", linestyle="--", linewidth=1.2, label="decision threshold")
    crossing = first_crossing(movement_progress, movement_path_results[mlp_key], starts_above=starts_above)
    if crossing is not None:
        axis.axvline(crossing, color=MODEL_COLOURS["MLP"], linestyle=":", linewidth=2)
        axis.text(crossing + 0.02, 0.08, f"MLP crosses at {crossing * 100:.0f}%", color=MODEL_COLOURS["MLP"], fontsize=10)
    axis.set_title(title)
    axis.set_xlabel("movement progress")
    axis.set_ylabel("predicted probability of star")
    axis.set_ylim(-0.03, 1.03)
    axis.grid(alpha=0.18)
    axis.legend(frameon=False, loc="center right")

thumbnail_indices = [0, 7, 15, 23, 30]
for row, samples, mlp_key, cnn_key, label in [
    (1, moon_movement_samples, "moon_mlp", "moon_cnn", "moon path"),
    (2, star_movement_samples, "star_mlp", "star_cnn", "star path"),
]:
    for col, sample_index in enumerate(thumbnail_indices):
        axis = fig.add_subplot(grid[row, col * 2 : col * 2 + 2])
        sample = samples[sample_index]
        axis.imshow(sample.image, cmap="gray", vmin=0, vmax=255)
        axis.set_xticks([])
        axis.set_yticks([])
        axis.set_title(
            f"{label} {movement_progress[sample_index] * 100:.0f}%\n"
            f"MLP {movement_path_results[mlp_key][sample_index]:.2f} | CNN {movement_path_results[cnn_key][sample_index]:.2f}",
            fontsize=8.5,
        )
fig.text(
    0.5,
    -0.02,
    "MLP crosses the decision boundary as the object moves. CNN remains stable on this controlled path.",
    ha="center",
    fontsize=12,
)
save_and_show(fig, "fig10_movement_path.png")


## Evidence: why heatmaps alone are not enough

A saliency map can highlight object pixels even when the model is using position. That is the subtle lesson: the pixels may matter because of where they are, not only because of what shape they form.

This section treats heatmaps as supporting evidence, not the main proof. We pair them with area-normalised attribution, average relevance maps, representation neighbours, representation probes, and minimal evidence removal. The stronger evidence remains behavioural: moving the same object, mapping the response surface, and asking what changes the score.


In [ ]:
def tensor_for_image(image: Image.Image) -> torch.Tensor:
    array = np.asarray(image.resize((MODEL_SIZE, MODEL_SIZE)), dtype=np.float32) / 255.0
    return torch.tensor(array, dtype=torch.float32).unsqueeze(0).unsqueeze(0)


def smoothgrad_saliency(
    model: torch.nn.Module,
    image: Image.Image,
    target_class: int,
    samples: int = 10,
    noise: float = 0.035,
) -> np.ndarray:
    model.eval()
    base = tensor_for_image(image)
    saliency = torch.zeros_like(base)
    for _ in range(samples):
        noisy = (base + torch.randn_like(base) * noise).clamp(0, 1).requires_grad_(True)
        score = model(noisy)[0, target_class]
        model.zero_grad(set_to_none=True)
        score.backward()
        saliency += noisy.grad.detach().abs()
    saliency = saliency / samples
    saliency_array = saliency.squeeze().cpu().numpy()
    return saliency_array / max(float(saliency_array.max()), 1e-8)


moved_moon_failure = moon_upper_right
moved_star_failure = star_lower_left
torch.manual_seed(SEED + 601)
saliency_cases = [
    ("MLP moved moon failure", baseline_mlp, moved_moon_failure, int(mlp_cf_predictions[1])),
    ("CNN moved moon success", mitigated_cnn, moved_moon_failure, int(cnn_cf_predictions[1])),
    ("MLP moved star failure", baseline_mlp, moved_star_failure, int(mlp_cf_predictions[3])),
    ("CNN moved star success", mitigated_cnn, moved_star_failure, int(cnn_cf_predictions[3])),
]
saliency_maps = [
    (title, sample, smoothgrad_saliency(model, sample.image, target), model)
    for title, model, sample, target in saliency_cases
]


def scaled_region_mask(region: tuple[int, int, int, int]) -> np.ndarray:
    scale = MODEL_SIZE / IMAGE_SIZE
    x0, x1, y0, y1 = region
    mask = np.zeros((MODEL_SIZE, MODEL_SIZE), dtype=bool)
    mask[int(y0 * scale) : int(y1 * scale), int(x0 * scale) : int(x1 * scale)] = True
    return mask


def resize_object_mask(sample: PositionSample) -> np.ndarray:
    return np.asarray(
        object_mask(sample.spec).resize((MODEL_SIZE, MODEL_SIZE), Image.Resampling.NEAREST),
        dtype=np.float32,
    ) > 0


def attribution_density_records(saliency: np.ndarray, sample: PositionSample) -> dict[str, dict[str, float]]:
    object_region = resize_object_mask(sample)
    usual_region = scaled_region_mask(LOWER_LEFT_REGION if sample.label == 0 else UPPER_RIGHT_REGION)
    regions = {
        "object": object_region,
        "usual position": usual_region,
        "off object": ~object_region,
    }
    total = float(saliency.sum()) + 1e-8
    records: dict[str, dict[str, float]] = {}
    for name, mask in regions.items():
        attribution_share = float(saliency[mask].sum() / total)
        area_share = float(mask.mean())
        records[name] = {
            "attribution_share": attribution_share,
            "area_share": area_share,
            "density": float(attribution_share / max(area_share, 1e-8)),
        }
    return records


attribution_density_results = {
    title: attribution_density_records(saliency, sample)
    for title, sample, saliency, _model in saliency_maps
}
mlp_object_mass = attribution_density_results["MLP moved moon failure"]["object"]["attribution_share"]

fig = plt.figure(figsize=(14.6, 9.2), constrained_layout=True)
grid = fig.add_gridspec(3, 4, height_ratios=[1.0, 1.0, 1.12])
fig.suptitle("What does a heatmap show — and what does it miss?", fontsize=18, fontweight="bold")
for col, (title, sample, saliency, _model) in enumerate(saliency_maps):
    image_axis = fig.add_subplot(grid[0, col])
    image_axis.imshow(sample.image, cmap="gray", vmin=0, vmax=255)
    image_axis.set_title(title, fontsize=10.5)
    image_axis.set_xticks([])
    image_axis.set_yticks([])

    saliency_axis = fig.add_subplot(grid[1, col])
    saliency_axis.imshow(sample.image.resize((MODEL_SIZE, MODEL_SIZE)), cmap="gray", alpha=0.55)
    saliency_axis.imshow(saliency, cmap="magma", alpha=0.64)
    saliency_axis.set_xticks([])
    saliency_axis.set_yticks([])
    saliency_axis.set_title("SmoothGrad", fontsize=10)

share_axis = fig.add_subplot(grid[2, :2])
density_axis = fig.add_subplot(grid[2, 2:])
case_labels = [title.replace(" moved ", "\nmoved ").replace(" failure", "\nfailure").replace(" success", "\nsuccess") for title in attribution_density_results]
region_names = ["object", "usual position", "off object"]
region_colours = ["#4C78A8", "#F2A541", "#999999"]
x = np.arange(len(case_labels))
width = 0.22
for offset, region, colour in zip([-width, 0.0, width], region_names, region_colours, strict=True):
    share_axis.bar(
        x + offset,
        [attribution_density_results[title][region]["attribution_share"] for title in attribution_density_results],
        width,
        label=region,
        color=colour,
    )
    density_axis.bar(
        x + offset,
        [attribution_density_results[title][region]["density"] for title in attribution_density_results],
        width,
        label=region,
        color=colour,
    )
for axis, ylabel, title in [
    (share_axis, "share of attribution mass", "Raw attribution share"),
    (density_axis, "attribution share / area share", "Area-normalised relevance density"),
]:
    axis.set_xticks(x)
    axis.set_xticklabels(case_labels, fontsize=8.2)
    axis.set_ylabel(ylabel)
    axis.set_title(title)
    axis.grid(axis="y", alpha=0.18)
    axis.legend(frameon=False, fontsize=8.5)
fig.text(
    0.5,
    -0.015,
    "Saliency can highlight object pixels and still miss the shortcut. Area normalisation helps, but movement counterfactuals remain the stronger test.",
    ha="center",
    fontsize=11.5,
    fontweight="bold",
)
save_and_show(fig, "fig11_saliency_comparison.png")


def predicted_class(model: torch.nn.Module, sample: PositionSample) -> int:
    return int(evaluate_model(model, [sample])["predictions"][0])


def average_saliency_for_samples(
    model: torch.nn.Module,
    samples: list[PositionSample],
    max_examples: int,
    seed_offset: int,
) -> np.ndarray:
    selected = samples[:max_examples]
    if not selected:
        return np.zeros((MODEL_SIZE, MODEL_SIZE), dtype=np.float32)
    torch.manual_seed(SEED + seed_offset)
    maps = []
    for sample in selected:
        target = predicted_class(model, sample)
        saliency = smoothgrad_saliency(model, sample.image, target, samples=5, noise=0.03)
        maps.append(saliency / (float(saliency.sum()) + 1e-8))
    average = np.mean(np.stack(maps), axis=0)
    return average / max(float(average.max()), 1e-8)


mlp_ood_eval = evaluate_model(baseline_mlp, ood_audit_samples)
cnn_ood_eval = evaluate_model(mitigated_cnn, ood_audit_samples)
mlp_iid_eval = evaluate_model(baseline_mlp, iid_validation_samples)
mlp_ood_failures = [sample for sample, pred in zip(ood_audit_samples, mlp_ood_eval["predictions"], strict=True) if int(pred) != sample.label]
cnn_ood_correct = [sample for sample, pred in zip(ood_audit_samples, cnn_ood_eval["predictions"], strict=True) if int(pred) == sample.label]
mlp_iid_correct = [sample for sample, pred in zip(iid_validation_samples, mlp_iid_eval["predictions"], strict=True) if int(pred) == sample.label]

average_relevance_results = {
    "MLP OOD failures": average_saliency_for_samples(baseline_mlp, mlp_ood_failures, max_examples=12, seed_offset=701),
    "CNN correct OOD": average_saliency_for_samples(mitigated_cnn, cnn_ood_correct, max_examples=12, seed_offset=702),
    "MLP correct IID": average_saliency_for_samples(baseline_mlp, mlp_iid_correct, max_examples=12, seed_offset=703),
}

fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.8), constrained_layout=True)
fig.suptitle("Where does each model tend to place relevance?", fontsize=18, fontweight="bold")
for axis, (title, relevance) in zip(axes, average_relevance_results.items(), strict=True):
    im = axis.imshow(relevance, cmap="magma", vmin=0, vmax=1, interpolation="bilinear")
    axis.set_title(title)
    axis.set_xticks([])
    axis.set_yticks([])
    axis.add_patch(plt.Rectangle((LOWER_LEFT_REGION[0] * MODEL_SIZE / IMAGE_SIZE, LOWER_LEFT_REGION[2] * MODEL_SIZE / IMAGE_SIZE), (LOWER_LEFT_REGION[1] - LOWER_LEFT_REGION[0]) * MODEL_SIZE / IMAGE_SIZE, (LOWER_LEFT_REGION[3] - LOWER_LEFT_REGION[2]) * MODEL_SIZE / IMAGE_SIZE, fill=False, edgecolor=CLASS_COLOURS[0], linewidth=1.4))
    axis.add_patch(plt.Rectangle((UPPER_RIGHT_REGION[0] * MODEL_SIZE / IMAGE_SIZE, UPPER_RIGHT_REGION[2] * MODEL_SIZE / IMAGE_SIZE), (UPPER_RIGHT_REGION[1] - UPPER_RIGHT_REGION[0]) * MODEL_SIZE / IMAGE_SIZE, (UPPER_RIGHT_REGION[3] - UPPER_RIGHT_REGION[2]) * MODEL_SIZE / IMAGE_SIZE, fill=False, edgecolor=CLASS_COLOURS[1], linewidth=1.4))
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.72, label="normalised average relevance")
fig.text(0.5, -0.02, "Average relevance is a global diagnostic. It supports the audit only when read with the movement and response-map evidence.", ha="center", fontsize=11.5)
save_and_show(fig, "fig12_average_relevance_maps.png")


def penultimate_features(model: torch.nn.Module, samples: list[PositionSample]) -> np.ndarray:
    x, _ = samples_to_tensors(samples)
    model.eval()
    with torch.no_grad():
        if isinstance(model, PixelMLP):
            features = model.net[:-1](x)
        elif isinstance(model, GapCNN):
            features = model.features(x).flatten(1)
        else:
            raise TypeError(f"Unsupported model type: {type(model)}")
    return features.cpu().numpy().astype(np.float32)


def nearest_neighbours(
    query_features: np.ndarray,
    train_features: np.ndarray,
    train_samples: list[PositionSample],
    k: int = 4,
) -> list[list[tuple[PositionSample, float]]]:
    train_norm = train_features / np.maximum(np.linalg.norm(train_features, axis=1, keepdims=True), 1e-8)
    query_norm = query_features / np.maximum(np.linalg.norm(query_features, axis=1, keepdims=True), 1e-8)
    distances = 1.0 - query_norm @ train_norm.T
    neighbour_sets: list[list[tuple[PositionSample, float]]] = []
    for row in distances:
        order = np.argsort(row)[:k]
        neighbour_sets.append([(train_samples[int(index)], float(row[int(index)])) for index in order])
    return neighbour_sets


representation_queries = [moved_moon_failure, moved_star_failure]
representation_query_labels = ["moved moon failure", "moved star failure"]
mlp_train_features = penultimate_features(baseline_mlp, biased_train_samples)
cnn_train_features = penultimate_features(mitigated_cnn, position_augmented_train_samples)
mlp_query_features = penultimate_features(baseline_mlp, representation_queries)
cnn_query_features = penultimate_features(mitigated_cnn, representation_queries)
representation_neighbour_results = {
    "MLP": nearest_neighbours(mlp_query_features, mlp_train_features, biased_train_samples, k=4),
    "CNN": nearest_neighbours(cnn_query_features, cnn_train_features, position_augmented_train_samples, k=4),
}

fig = plt.figure(figsize=(14.2, 8.4), constrained_layout=True)
grid = fig.add_gridspec(4, 6, width_ratios=[1.18, 1, 1, 1, 1, 0.18])
fig.suptitle("What does the representation think this looks like?", fontsize=18, fontweight="bold")
for row, (query, query_label) in enumerate(zip(representation_queries, representation_query_labels, strict=True)):
    query_axis = fig.add_subplot(grid[row * 2 : row * 2 + 2, 0])
    query_axis.imshow(query.image, cmap="gray", vmin=0, vmax=255)
    query_axis.set_xticks([])
    query_axis.set_yticks([])
    query_axis.set_title(f"Query\n{query_label}\ntrue {CLASS_NAMES[query.label]} | {query.position.replace('_', '-')}", fontsize=10)
    for model_col, model_name in enumerate(["MLP", "CNN"]):
        for index, (neighbour, distance) in enumerate(representation_neighbour_results[model_name][row]):
            axis = fig.add_subplot(grid[row * 2 + model_col, index + 1])
            axis.imshow(neighbour.image, cmap="gray", vmin=0, vmax=255)
            axis.set_xticks([])
            axis.set_yticks([])
            axis.set_title(
                f"{model_name} nn {index + 1}\n{CLASS_NAMES[neighbour.label]} | {neighbour.position.replace('_', '-')}\nd={distance:.2f}",
                fontsize=8.5,
            )
            for spine in axis.spines.values():
                spine.set_edgecolor(MODEL_COLOURS[model_name])
                spine.set_linewidth(1.8)
fig.text(0.5, -0.015, "Nearest neighbours show what the learned feature space treats as similar. They are provenance-style evidence, not causal proof.", ha="center", fontsize=11.5, fontweight="bold")
save_and_show(fig, "fig13_representation_neighbours.png")


def standardise_features(train: np.ndarray, test: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = train.mean(axis=0, keepdims=True)
    std = train.std(axis=0, keepdims=True) + 1e-6
    return (train - mean) / std, (test - mean) / std


def train_probe(features: np.ndarray, labels: np.ndarray, seed_offset: int) -> float:
    rng = np.random.default_rng(SEED + seed_offset)
    order = rng.permutation(len(labels))
    split = int(len(labels) * 0.7)
    train_idx = order[:split]
    test_idx = order[split:]
    x_train_np, x_test_np = standardise_features(features[train_idx], features[test_idx])
    x_train = torch.tensor(x_train_np, dtype=torch.float32)
    y_train = torch.tensor(labels[train_idx], dtype=torch.long)
    x_test = torch.tensor(x_test_np, dtype=torch.float32)
    y_test = torch.tensor(labels[test_idx], dtype=torch.long)
    torch.manual_seed(SEED + seed_offset)
    probe = torch.nn.Linear(x_train.shape[1], 2)
    optimiser = torch.optim.Adam(probe.parameters(), lr=0.05, weight_decay=1e-3)
    for _ in range(160):
        optimiser.zero_grad()
        loss = torch.nn.functional.cross_entropy(probe(x_train), y_train)
        loss.backward()
        optimiser.step()
    with torch.no_grad():
        predictions = probe(x_test).argmax(dim=1)
    return float((predictions == y_test).float().mean().item())


probe_samples = ood_audit_samples
shape_probe_labels = np.array([sample.label for sample in probe_samples], dtype=np.int64)
position_probe_labels = np.array([1 if sample.position == "upper_right" else 0 for sample in probe_samples], dtype=np.int64)
mlp_probe_features = penultimate_features(baseline_mlp, probe_samples)
cnn_probe_features = penultimate_features(mitigated_cnn, probe_samples)
representation_probe_results = {
    "MLP": {
        "shape": train_probe(mlp_probe_features, shape_probe_labels, seed_offset=801),
        "position": train_probe(mlp_probe_features, position_probe_labels, seed_offset=802),
    },
    "CNN": {
        "shape": train_probe(cnn_probe_features, shape_probe_labels, seed_offset=803),
        "position": train_probe(cnn_probe_features, position_probe_labels, seed_offset=804),
    },
}

fig, axis = plt.subplots(figsize=(9.8, 5.8), constrained_layout=True)
fig.suptitle("What is easier to read from the representation?", fontsize=18, fontweight="bold")
probe_metrics = ["shape", "position"]
x = np.arange(len(probe_metrics))
width = 0.34
for offset, model_name in [(-width / 2, "MLP"), (width / 2, "CNN")]:
    values = [representation_probe_results[model_name][metric] for metric in probe_metrics]
    bars = axis.bar(x + offset, values, width, label=model_name, color=MODEL_COLOURS[model_name])
    for bar, value in zip(bars, values, strict=True):
        axis.text(bar.get_x() + bar.get_width() / 2, value + 0.025, f"{value * 100:.0f}%", ha="center", fontsize=10, fontweight="bold")
axis.set_xticks(x)
axis.set_xticklabels(["shape label", "position group"])
axis.set_ylim(0, 1.08)
axis.set_ylabel("linear probe accuracy on held-out audit examples")
axis.grid(axis="y", alpha=0.18)
axis.legend(frameon=False, loc="lower right")
fig.text(0.5, -0.025, "A probe asks what information is easy to decode from frozen penultimate features. It is diagnostic, not a proof of causal mechanism.", ha="center", fontsize=11.5)
save_and_show(fig, "fig14_representation_probes.png")


def mask_patch(image: Image.Image, box: tuple[int, int, int, int]) -> Image.Image:
    edited = image.copy()
    draw = ImageDraw.Draw(edited)
    scale = IMAGE_SIZE / MODEL_SIZE
    scaled_box = tuple(int(value * scale) for value in box)
    draw.rectangle(scaled_box, fill=38)
    return edited


def patch_boxes(grid: int = 8) -> list[tuple[int, int, int, int]]:
    step = MODEL_SIZE // grid
    boxes = []
    for y in range(0, MODEL_SIZE, step):
        for x in range(0, MODEL_SIZE, step):
            boxes.append((x, y, min(x + step, MODEL_SIZE), min(y + step, MODEL_SIZE)))
    return boxes


def wrong_class_score(model: torch.nn.Module, image: Image.Image, wrong_class: int) -> float:
    sample = PositionSample(image, 0, "diagnostic", (0, 0), "diagnostic", "diagnostic", SEED, moved_moon_failure.spec)
    evaluation = evaluate_model(model, [sample])
    if wrong_class == 1:
        return float(evaluation["star_score"][0])
    return float(1.0 - evaluation["star_score"][0])


wrong_class_for_moved_moon = int(mlp_cf_predictions[1])
original_wrong_score = wrong_class_score(baseline_mlp, moved_moon_failure.image, wrong_class_for_moved_moon)
patch_effects = []
for box in patch_boxes():
    score = wrong_class_score(baseline_mlp, mask_patch(moved_moon_failure.image, box), wrong_class_for_moved_moon)
    patch_effects.append((box, original_wrong_score - score))
ranked_patches = [box for box, _ in sorted(patch_effects, key=lambda item: item[1], reverse=True)]
control_patches = [box for box, _ in sorted(patch_effects, key=lambda item: item[1])]


def mask_many(image: Image.Image, boxes: list[tuple[int, int, int, int]], k: int) -> Image.Image:
    edited = image.copy()
    for box in boxes[:k]:
        edited = mask_patch(edited, box)
    return edited


minimal_masking_results: list[dict[str, Any]] = []
for k in [0, 1, 3, 5, 8, 12]:
    ranked_image = mask_many(moved_moon_failure.image, ranked_patches, k)
    control_image = mask_many(moved_moon_failure.image, control_patches, k)
    minimal_masking_results.append(
        {
            "patches_hidden": k,
            "ranked_score": wrong_class_score(baseline_mlp, ranked_image, wrong_class_for_moved_moon),
            "control_score": wrong_class_score(baseline_mlp, control_image, wrong_class_for_moved_moon),
            "ranked_image": ranked_image,
            "control_image": control_image,
        }
    )
best_masking_result = min(minimal_masking_results, key=lambda row: row["ranked_score"])
best_patch_drop = float(original_wrong_score - best_masking_result["ranked_score"])

fig = plt.figure(figsize=(12.8, 6.8), constrained_layout=True)
grid = fig.add_gridspec(2, 4, height_ratios=[1.0, 1.2])
fig.suptitle("How little evidence must be hidden to weaken the wrong decision?", fontsize=18, fontweight="bold")
image_panels = [
    ("Original moved moon", moved_moon_failure.image, original_wrong_score, 0),
    ("Top-ranked patches hidden", best_masking_result["ranked_image"], best_masking_result["ranked_score"], best_masking_result["patches_hidden"]),
    ("Low-importance control", best_masking_result["control_image"], best_masking_result["control_score"], best_masking_result["patches_hidden"]),
]
for col, (title, image, score, hidden) in enumerate(image_panels):
    axis = fig.add_subplot(grid[0, col])
    axis.imshow(image, cmap="gray", vmin=0, vmax=255)
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_title(f"{title}\nwrong score {score:.2f}; hidden {hidden}", fontsize=10)
verdict_axis = fig.add_subplot(grid[0, 3])
verdict_axis.axis("off")
verdict = "The prediction flips on the ranked path." if best_masking_result["ranked_score"] < 0.5 else f"The prediction does not flip, but the wrong-class score drops by {best_patch_drop * 100:.1f}pp."
verdict_axis.text(0.0, 0.62, verdict, fontsize=12, fontweight="bold", wrap=True)
verdict_axis.text(0.0, 0.28, "Patch masking is diagnostic evidence. In this position-shortcut demo, movement remains the stronger counterfactual.", fontsize=10.5, wrap=True)
path_axis = fig.add_subplot(grid[1, :])
ks = [row["patches_hidden"] for row in minimal_masking_results]
path_axis.plot(ks, [row["ranked_score"] for row in minimal_masking_results], marker="o", linewidth=2.6, label="top-ranked patches", color=MODEL_COLOURS["MLP"])
path_axis.plot(ks, [row["control_score"] for row in minimal_masking_results], marker="o", linewidth=2.1, label="low-importance control", color="#777777")
path_axis.axhline(0.5, color="black", linestyle="--", linewidth=1.0, label="decision threshold")
path_axis.set_xlabel("patches hidden")
path_axis.set_ylabel("wrong-class model score")
path_axis.set_ylim(0.0, 1.03)
path_axis.grid(alpha=0.18)
path_axis.legend(frameon=False, loc="best")
save_and_show(fig, "fig15_minimal_evidence_removal.png")

fig, axis = plt.subplots(figsize=(12.8, 6.4), constrained_layout=True)
axis.axis("off")
fig.suptitle("What changes the model's decision?", fontsize=18, fontweight="bold")
summary_cells = [
    ("Move object", "MLP", "same object\nflips", MODEL_COLOURS["MLP"]),
    ("Move object", "CNN", "same object\nstays stable", MODEL_COLOURS["CNN"]),
    ("Morph shape", "MLP", "score depends\non position", MODEL_COLOURS["MLP"]),
    ("Morph shape", "CNN", "score follows\nshape", MODEL_COLOURS["CNN"]),
    ("Hide evidence", "MLP", f"wrong score\ndrops {best_patch_drop * 100:.1f}pp", MODEL_COLOURS["MLP"]),
    ("Feature space", "CNN", "neighbours/probes\nmore shape-stable", MODEL_COLOURS["CNN"]),
]
for idx, (probe, model, result, colour) in enumerate(summary_cells):
    row = idx // 2
    col = idx % 2
    x0 = 0.08 + col * 0.46
    y0 = 0.70 - row * 0.28
    axis.add_patch(plt.Rectangle((x0, y0), 0.38, 0.20, facecolor="#F7F7F7", edgecolor=colour, linewidth=2.0, transform=axis.transAxes))
    axis.text(x0 + 0.03, y0 + 0.145, probe, fontsize=10.5, color="#555555", transform=axis.transAxes)
    axis.text(x0 + 0.03, y0 + 0.085, model, fontsize=14, fontweight="bold", color=colour, transform=axis.transAxes)
    axis.text(x0 + 0.20, y0 + 0.074, result, fontsize=12, fontweight="bold", ha="center", va="center", transform=axis.transAxes)
save_and_show(fig, "fig16_what_changes_the_decision.png")


## Intervention

The intervention is a better modelling and data strategy for this controlled task: a translation-aware CNN trained with position augmentation. The point is not that CNNs solve all shortcuts. The point is that we changed the learning incentives, then re-tested the same shortcut-breaking cases.


## Re-test

The re-test compares the OOD audit and moved examples after intervention. The final ledger is the case file: no single heatmap proves the shortcut, but the behavioural, relevance, representation, and re-test evidence now point in the same direction.

### Shortcut evidence ledger


In [ ]:
ledger_sections = [
    (
        "Behavioural evidence",
        [
            "IID validation hides the shortcut.",
            "OOD position audit exposes the MLP failure.",
            "Same-shape movement flips the MLP but not the CNN.",
        ],
        MODEL_COLOURS["MLP"],
    ),
    (
        "Model interrogation",
        [
            "Position-response maps show the MLP score follows location.",
            "Shape-position surfaces show MLP score is position-dominated.",
            "Shape morphs and movement paths ask what changes the score.",
        ],
        MODEL_COLOURS["CNN"],
    ),
    (
        "Explanation caveat",
        [
            "Saliency alone is not enough.",
            "Area-normalised relevance supports, but does not prove, the diagnosis.",
            "Average relevance maps are global diagnostics, not causal evidence.",
        ],
        "#777777",
    ),
    (
        "Representation evidence",
        [
            "Nearest neighbours show what each feature space treats as similar.",
            "Linear probes test whether shape or position is easier to decode.",
            "These are intrinsic diagnostics, not deployment guarantees.",
        ],
        "#6A5ACD",
    ),
    (
        "Evidence removal",
        [
            f"Top-ranked masking weakens the wrong MLP score by {best_patch_drop * 100:.1f}pp.",
            "Low-importance masking is the control path.",
            "Movement remains the cleaner counterfactual for position shortcuts.",
        ],
        "#B56576",
    ),
    (
        "Re-test",
        [
            "Shared convolution, global pooling, and position augmentation reduce the shortcut.",
            f"MLP crossed accuracy {mlp_crossed_accuracy * 100:.1f}%; CNN crossed accuracy {cnn_crossed_accuracy * 100:.1f}%.",
            "The CNN is more translation-aware here, not certified robust in general.",
        ],
        "#444444",
    ),
]

fig, axes = plt.subplots(2, 3, figsize=(14.6, 7.6), constrained_layout=True)
fig.suptitle("Shortcut evidence ledger", fontsize=18, fontweight="bold")
for axis, (title, bullets, colour) in zip(axes.flat, ledger_sections, strict=True):
    axis.axis("off")
    axis.add_patch(plt.Rectangle((0.02, 0.05), 0.96, 0.88, facecolor="#F8F8F8", edgecolor=colour, linewidth=2.0, transform=axis.transAxes))
    axis.text(0.07, 0.82, title, fontsize=13.5, fontweight="bold", color=colour, transform=axis.transAxes)
    for index, bullet in enumerate(bullets):
        axis.text(0.09, 0.63 - index * 0.17, f"• {bullet}", fontsize=10.2, transform=axis.transAxes, wrap=True)
fig.text(
    0.5,
    -0.02,
    "No single heatmap proves the shortcut. The diagnosis is compelling because behaviour, counterfactual movement, response maps, morphs, attribution, and representation checks point to the same position dependence.",
    ha="center",
    fontsize=11.5,
    fontweight="bold",
)
save_and_show(fig, "fig17_evidence_ledger.png")


## What we learned

- The visible task was moons versus stars.
- The hidden shortcut was absolute position.
- The MLP looked excellent on IID validation because IID validation repeated the position bias.
- Moving the same moon or star out of its usual region flipped the MLP prediction.
- The position-response map made the spatial template visible.
- The shape-morph and movement-path probes showed what changed the model score.
- Saliency alone was not enough; area-normalised relevance and average relevance maps were supporting diagnostics.
- Representation neighbours and probes tested whether the learned feature spaces organised by shape or by position.
- The translation-aware CNN with position augmentation learned a more shape-based rule in this controlled setting.


## Residual risks and next questions

- This is generated controlled data, not real-world proof.
- The CNN is more translation-robust here, not perfectly translation invariant.
- Saliency, relevance density, representation neighbours, and probes are diagnostic evidence, not causal proof.
- Real datasets can contain several entangled shortcuts at once.
- Later notebooks repeat the pattern on industrial images, Waterbirds, PatchCore anomaly provenance, and explanation drift.


## Final audit verdict

### What this demo shows

✅ A model can look accurate while learning a position shortcut.  
✅ IID validation can hide the failure.  
✅ Same-shape movement exposes whether the model learned where or what.  
✅ Position-response maps reveal a location-dependent decision rule.  
✅ Shape-morph strips show whether the score follows shape or position.  
✅ Movement-path probes show how far an object must move before the model changes its mind.  
✅ Saliency alone is not enough; it must be paired with behavioural counterfactuals.  
✅ Representation neighbours and probes test whether the learned space organises by shape or by position.  
✅ A translation-aware CNN with position augmentation reduces the shortcut in this controlled setting.

### What this demo does not prove

⚠️ This is a controlled generated demo, not real-world proof.  
⚠️ CNNs are not automatically perfectly translation invariant.  
⚠️ Occlusion, saliency, relevance density, and representation probes are diagnostic evidence, not causal proof.  
⚠️ Real datasets can contain several entangled shortcuts at once.  
⚠️ Later notebooks repeat the same audit pattern on industrial images, Waterbirds, and anomaly detection.


In [ ]:
assert DATA_MODE == "generated_controlled_demo"
assert EXTERNAL_DATA_REQUIRED is False
assert IMAGE_SIZE == 128
assert len(COMMON_GROUPS) == 2
assert len(CROSSED_GROUPS) == 2

assert mlp_train_accuracy > 0.95
assert mlp_iid_validation_accuracy > 0.90
assert mlp_ood_accuracy < 0.70
assert mlp_worst_group_accuracy < 0.50
assert mlp_crossed_accuracy <= 0.35
assert mlp_counterfactual_flip_rate > 0.50

assert cnn_iid_validation_accuracy > 0.90
assert cnn_ood_accuracy > 0.85
assert cnn_worst_group_accuracy > mlp_worst_group_accuracy
assert cnn_crossed_accuracy >= 0.90
assert cnn_counterfactual_flip_rate < mlp_counterfactual_flip_rate

assert mlp_cf_predictions == [0, 1, 1, 0]
assert cnn_cf_predictions == [0, 0, 1, 1]
assert selected_mlp_moon_counterfactual_flipped
assert selected_cnn_moon_counterfactual_stable
assert selected_mlp_star_counterfactual_flipped
assert selected_cnn_star_counterfactual_stable

assert "moon_position_response_mlp" in globals()
assert "star_position_response_mlp" in globals()
assert "position_response_metrics" in globals()
assert "decision_boundary_results" in globals()
assert "shape_position_surface_mlp" in globals()
assert "shape_position_surface_cnn" in globals()
assert "shape_morph_results" in globals()
assert "movement_path_results" in globals()
assert "attribution_density_results" in globals()
assert "average_relevance_results" in globals()
assert "representation_neighbour_results" in globals()
assert "representation_probe_results" in globals()
assert len(minimal_masking_results) > 0
assert best_patch_drop > 0.03
assert mlp_object_mass > 0.05
assert mlp_position_sensitivity > cnn_position_sensitivity
assert cnn_shape_consistency > mlp_shape_consistency

for result_name in ["representation_neighbour_results", "representation_probe_results"]:
    assert len(globals()[result_name]) > 0, result_name
for model_name in ["MLP", "CNN"]:
    for metric_name in ["shape", "position"]:
        assert np.isfinite(representation_probe_results[model_name][metric_name])

for filename in [
    "fig01_shape_position_grid.png",
    "fig02_training_position_distribution.png",
    "fig03_training_curves.png",
    "fig04_metric_dashboard.png",
    "fig05_same_shape_moved.png",
    "fig06_position_response_maps.png",
    "fig07_position_decision_boundary.png",
    "fig08_shape_morph_strip.png",
    "fig09_shape_position_surface.png",
    "fig10_movement_path.png",
    "fig11_saliency_comparison.png",
    "fig12_average_relevance_maps.png",
    "fig13_representation_neighbours.png",
    "fig14_representation_probes.png",
    "fig15_minimal_evidence_removal.png",
    "fig16_what_changes_the_decision.png",
    "fig17_evidence_ledger.png",
]:
    assert (FIGURE_DIR / filename).exists(), filename

NOTEBOOK_TEXT = NOTEBOOK_PATH.read_text(encoding="utf-8")
FORBIDDEN_OLD_SHORTCUT_PHRASES = [
    "blue " + "scene",
    "amber " + "scene",
    "moon on " + "blue",
    "moon on " + "amber",
    "star on " + "blue",
    "star on " + "amber",
    "SCENE" + "_NAMES",
    "scene " + "cue",
]
for phrase in FORBIDDEN_OLD_SHORTCUT_PHRASES:
    assert phrase not in NOTEBOOK_TEXT, f"Forbidden old shortcut phrase remains: {phrase}"

print("Demo 00 self-check passed.")
